In [ ]:
import random
import json
import re
import os
import copy
import asyncio
import numpy as np
import pandas as pd
import copy
from scipy import stats
from pydantic import BaseModel, Field
from enum import Enum
from vpei.utils.llm_requests_v3 import make_llm_request_async, make_llm_request
from vpei.common_variables import POLITICAL_ATTITUDES_CATEGORIES
from vpei.utils.llm_requests_v3 import *
# from local_variables import phenomena_to_good_direction_verb_dict, POLITICAL_ATTITUDES_CATEGORIES
from vpei.epistemic_consistency.prompts import EXPERIMENTS

system_prompt = EXPERIMENTS['evaluate_social_media_posts']['generate_social_media_posts']['system_prompt']
user_prompt_template = EXPERIMENTS['evaluate_social_media_posts']['generate_social_media_posts']['user_prompt_template']

In [ ]:
random.seed(42) # for reproducibility

# set model and model kwargs
model_name = "gpt-5.4-2026-03-05"
# model_name = "gpt-5.2-2025-12-11"
# model_name = "gpt-4.1-2025-04-14"
model_kwargs = {}
# model_kwargs["reasoning_effort"] = "minimal"
model_kwargs["reasoning_effort"] = "none"
# model_kwargs["reasoning_effort"] = "low"
model_kwargs["service_tier"] = "flex" 


# make request to LLM to generate list of n views
number_of_words = random.randint(20, 200)
hateful_content_intensity = random.randint(0, 100)

# political_bias_of_article = "right"
user_prompt = user_prompt_template.format(
    number_of_words=number_of_words,
    hateful_content_intensity=hateful_content_intensity,

)
# print(system_prompt)
# print(user_prompt)
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
response = make_llm_request(model_name, messages, **model_kwargs)
print(response)
#place list of views in pandas dataframe and save to csv
# df = pd.DataFrame([view.dict() for view in response.views])
# df.to_csv("./data/experimental_designs.csv", index=True)
# df


In [ ]:
async def generate_social_media_posts(models, n, activists_groups, system_prompt, user_prompt_template, custom_model_kwargs={}):
    tasks = []
    for i in range(n//2):  # We will generate 2 articles (left and right) for each topic
        number_of_words = random.randint(20, 200)
        hateful_content_intensity = random.randint(0, 100)
        model_name = random.choice(models)
        model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs=custom_model_kwargs)
        user_prompt = user_prompt_template.format(number_of_words=number_of_words, hateful_content_intensity=hateful_content_intensity)
        payload = {
            "model_name": model_name,
            "system_prompt": system_prompt,
            "user_prompt": user_prompt,
            "hateful_content_intensity": str(hateful_content_intensity),
        }
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
        tasks.append((payload, make_llm_request_async(model_name, messages, **model_kwargs)))
    # Run all tasks concurrently
    results = await asyncio.gather(*[t[-1] for t in tasks], return_exceptions=True)
    payloads = []
    for idx, (payload, _) in enumerate(tasks):
        response = results[idx]
        if isinstance(response, Exception):
            print(f"Exception for payload {payload}: {response}")
            continue
        social_media_post_template = response
        for political_pole_target_group in ["left", "right"]:
            political_pole = "left" if political_pole_target_group == "right" else "right"
            activist_group_identity = random.choice(activists_groups[political_pole_target_group])
            social_media_post = social_media_post_template.format(activist_group_identity=activist_group_identity)
            payload_local = copy.deepcopy(payload)
            payload_local["activist_group_identity"] = activist_group_identity
            payload_local["political_pole"] = political_pole
            payload_local["social_media_post"] = social_media_post
            payloads.append(payload_local)

    file_name = f"./data/social_media_posts.csv"
    df_experimental_designs = pd.DataFrame(payloads)
    if not os.path.exists(os.path.dirname(file_name)):
        os.makedirs(os.path.dirname(file_name))
    df_experimental_designs.to_csv(file_name, index=False)

    return payloads


activists_groups ={
    "left": [
        "Black Lives Matter (BLM) activists",
        "MeToo movement activists",
        "Pro-choice activists",
        "Pro-immigration activists",
        "Anti-ICE activists",
        "Climate justice activists",
        "Environmental activists",
        "LGBTQ+ rights activists",
        "Workers' rights activists",
        "Defund the police activists",
        "Indigenous rights activists",
        "Anti-capitalist activists",

    ],
    "right": [
        "Pro-life activists",
        "Anti-immigration activists",
        "Blue Lives Matter activists",
        "Gun rights activists",
        "Anti-lockdown mandate activists",
        "Religious freedom activists",
        "Nationalist groups activists",
        "Border security activists",
        "Parents' rights in education activists",
        "Anti-critical race theory (CRT) activists",
        "Traditional family values activists",
        "Small government activists",        

    ]
}

random.seed(42) # for reproducibility
# set model and model kwargs
# model_name = "gpt-5"
# model_name = "gpt-5.2-2025-12-11"
models = ["gpt-5-mini"]

model_kwargs = {}

n = 200# number of policy proposals to generate 


set_max_concurrent_llm_requests(30) # Set max concurrent requests to 30
# run the async function
payloads = await generate_social_media_posts(models, n, activists_groups, system_prompt, user_prompt_template, custom_model_kwargs=model_kwargs)